In [129]:
from google.colab import files

uploaded = files.upload()


Saving follow_up.xls to follow_up (3).xls


In [130]:
from google.colab import files

uploaded = files.upload()


Saving appointments_clean.csv to appointments_clean (3).csv


In [131]:
from google.colab import files

uploaded = files.upload()


Saving test_risk_scores.csv to test_risk_scores (3).csv


In [132]:
# install libraries
!pip install -q google-genai pydantic

In [133]:
import pandas as pd
import json
import time

from google import genai
from google.genai import types

from pydantic import BaseModel, Field, ValidationError

In [134]:
API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client created successfully!")

Paste your Gemini API key: AQ.Ab8RN6LXqBG2VUzMi16zKILO2k26visO0p4q14-LpseHDEfnaw
Gemini client created successfully!


In [135]:
# Update to the newest recommended model to resolve the 404 restriction
MODEL_NAME = "gemini-3.8-flash"

In [136]:
# Define the JSON response schema
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [137]:
# Define the allowed root causes
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [138]:
# Create the CRAFT prompt
SYSTEM_INSTRUCTION = """
You are a careful patient follow-up analyst.
Analyse patient notes objectively.
Do not invent information or assume facts that are not stated.
Return only valid JSON matching the supplied schema.
"""


def create_craft_prompt(note_text):

    prompt = f"""
    CONTEXT:
    CareLoop Clinics wants to understand why patients miss appointments.

    ROLE:
    You are an experienced patient follow-up data analyst.

    ACTION:
    Read the patient follow-up note and identify the main reason
    for missing the appointment.

    Allowed root causes:
    forgot_or_no_reminder
    distance_or_transport
    cost_or_fee
    illness_or_emergency
    work_or_schedule
    other

    If the reason is unclear, choose other.
    Do not guess or invent information.

    TONE:
    Be clear, neutral and professional.

    FORMAT:
    Return JSON with exactly these four fields:
    root_cause, confidence, summary, recommended_action.

    Confidence must be between 0 and 1.
    Summary must contain no more than 25 words.

    PATIENT NOTE:
    {note_text}
    """

    return prompt

In [139]:
# Create the analyse_note() function (Fixed config for Gemini 1.5)
API_CALLS = 0
def analyse_note(note_text):
    global API_CALLS
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=create_craft_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            max_output_tokens=1200
        )
    )

    API_CALLS += 1
    data = json.loads(response.text)
    return data

In [140]:
# Test the function using N001
notes_df = pd.read_csv("follow_up.xls")
print(notes_df.head(10))

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
5    N006          A0507  Manager called an urgent meeting at the office...
6    N007          A0333  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
7    N008          A0222  City bus strike that day, no auto available ne...
8    N009          A0977  Phone was switched off, could not reach the pa...
9    N010          A0596  Had a shift change at the factory and could no...


In [144]:
# Select the first note:
note_001 = notes_df.iloc[0]["note_text"]
result = analyse_note(note_001)
print(result)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 22h56m0.457476768s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

In [145]:
# Task 4.2 — Validate and process the first 10 notes
# Create a Pydantic validation model
class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3)
  confidence: float = Field(ge=0, le=1)
  summary: str = Field(min_length=5)
  recommended_action: str = Field(min_length=5)

In [146]:
# Create a function to get the raw Gemini response
def get_raw_response(note_text):
  global API_CALLS
  response = client.models.generate_content(
        model=MODEL_NAME,
        contents=create_craft_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            max_output_tokens=1200
        )
    )

  API_CALLS += 1
  return response.text

In [147]:
# Read the first 10 notes
# Note: The uploaded file is named "follow_up.xls" (with an underscore) and is formatted as a CSV.
notes_df = pd.read_csv("follow_up.xls")
first_10_notes = notes_df.head(10).copy()
print("Number of notes:", len(first_10_notes))
print(first_10_notes)

Number of notes: 10
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
5    N006          A0507  Manager called an urgent meeting at the office...
6    N007          A0333  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
7    N008          A0222  City bus strike that day, no auto available ne...
8    N009          A0977  Phone was switched off, could not reach the pa...
9    N010          A0596  Had a shift change at the factory and could no...


In [148]:
# Process the notes one by one
raw_response = ""
data = {}
try:
    raw_response = get_raw_response(str(note_text))
    data = json.loads(raw_response)
    validated = FollowupInsight(**data)
    if validated.root_cause not in ALLOWED_CAUSES:
        raise ValueError("Root cause is not in the allowed list")
    validation_status = "valid"
except Exception as e:
    validation_status = "invalid"
    print("Validation failed:", e)
root_cause = data.get("root_cause", "")
confidence = data.get("confidence", "")
summary = data.get("summary", "")
recommended_action = data.get("recommended_action", "")

In [149]:
# Save the analysed notes
notes_analysed = pd.DataFrame(results)

columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

notes_analysed = notes_analysed[columns]

notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print(notes_analysed)

  note_id appointment_id root_cause confidence           summary  \
0    N001          A0872                                           
1    N002          A0897                                           
2    N003          A0526                                           
3    N004          A0926      other        0.0  No note recorded   
4    N005          A0534                                           
5    N006          A0507                                           

  recommended_action validation_status raw_response  
0                              invalid               
1                              invalid               
2                              invalid               
3    Re-call patient     skipped_blank               
4                              invalid               
5                              invalid               


In [151]:
# Check the results
print("Total notes:", len(notes_analysed))

print("\nValidation status:")
print(notes_analysed["validation_status"].value_counts())

print("\nRoot cause counts:")
print(notes_analysed["root_cause"].value_counts())

print("\nN004 result:")
print(
    notes_analysed[
        notes_analysed["note_id"] == "N004"
    ]
)

Total notes: 6

Validation status:
validation_status
invalid          5
skipped_blank    1
Name: count, dtype: int64

Root cause counts:
root_cause
         5
other    1
Name: count, dtype: int64

N004 result:
  note_id appointment_id root_cause confidence           summary  \
3    N004          A0926      other        0.0  No note recorded   

  recommended_action validation_status raw_response  
3    Re-call patient     skipped_blank               


In [152]:
# Task 4.3 — Test invalid responses and compare prompts
# Test an invalid confidence value
bad_response_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment",
    "recommended_action": "Send an appointment reminder"
}

try:

    test_1 = FollowupInsight(**bad_response_1)

    print(test_1)

except ValidationError as e:

    print("Validation Error:")
    print(e)

Validation Error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [153]:
# Test a missing field
bad_response_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.8,
    "summary": "Patient had transport difficulties"
}

try:

    test_2 = FollowupInsight(**bad_response_2)

    print(test_2)

except ValidationError as e:

    print("Validation Error:")
    print(e)

Validation Error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...transport difficulties'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [154]:
# Compare two prompts
def compare_prompt(note_text, prompt):

    global API_CALLS

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            max_output_tokens=1200,
            thinking_config=types.ThinkingConfig(
                thinking_level="MINIMAL"
            )
        )
    )

    API_CALLS += 1

    return json.loads(response.text)

In [155]:
# Extract N007 and N009
note_007 = notes_df.loc[
    notes_df["note_id"] == "N007",
    "note_text"
].iloc[0]

note_009 = notes_df.loc[
    notes_df["note_id"] == "N009",
    "note_text"
].iloc[0]

In [156]:
# Redefine compare_prompt with a clean fallback using correct model names
def compare_prompt(note_text, prompt):
    global API_CALLS
    # Use full paths for both models to ensure the v1beta endpoint can locate them correctly
    models_to_try = [MODEL_NAME, "gemini-1.5-flash"]

    for model in models_to_try:
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_INSTRUCTION,
                        temperature=0,
                        response_mime_type="application/json",
                        response_schema=SCHEMA,
                        max_output_tokens=1200
                    )
                )
                API_CALLS += 1
                return json.loads(response.text)
            except Exception as e:
                # If it's a 503 error, back off and retry
                if "503" in str(e):
                    if attempt < max_retries - 1:
                        wait_time = (attempt + 1) * 15
                        print(f"[{model}] API busy (503). Retrying in {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                        time.sleep(wait_time)
                    else:
                        if model != models_to_try[-1]:
                            print(f"[{model}] Failed after retries. Falling back to next model...")
                        else:
                            raise e
                else:
                    raise e

# Run V1 and V2 prompts
# V1: Simple prompt
print("Processing V1 for Note 007...")
v1_007 = compare_prompt(
    str(note_007),
    "Classify this note: " + str(note_007)
)

time.sleep(5)

print("Processing V1 for Note 009...")
v1_009 = compare_prompt(
    str(note_009),
    "Classify this note: " + str(note_009)
)

time.sleep(5)

# V2: CRAFT prompt
print("Processing V2 for Note 007...")
v2_007 = compare_prompt(
    str(note_007),
    create_craft_prompt(str(note_007))
)

time.sleep(5)

print("Processing V2 for Note 009...")
v2_009 = compare_prompt(
    str(note_009),
    create_craft_prompt(str(note_009))
)

print("Comparison successfully completed!")

Processing V1 for Note 007...
[gemini-3.8-flash] API busy (503). Retrying in 15s... (Attempt 1/3)


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 22h53m4.902747118s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

In [158]:
# Create the prompt comparison table
comparison_results = pd.DataFrame([
    {
        "Note": "N007",
        "Prompt": "V1 - Simple",
        "Root Cause": v1_007["root_cause"],
        "Allowed": v1_007["root_cause"] in ALLOWED_CAUSES
    },
    {
        "Note": "N007",
        "Prompt": "V2 - CRAFT",
        "Root Cause": v2_007["root_cause"],
        "Allowed": v2_007["root_cause"] in ALLOWED_CAUSES
    },
    {
        "Note": "N009",
        "Prompt": "V1 - Simple",
        "Root Cause": v1_009["root_cause"],
        "Allowed": v1_009["root_cause"] in ALLOWED_CAUSES
    },
    {
        "Note": "N009",
        "Prompt": "V2 - CRAFT",
        "Root Cause": v2_009["root_cause"],
        "Allowed": v2_009["root_cause"] in ALLOWED_CAUSES
    }
])

print(comparison_results)

NameError: name 'v2_007' is not defined

In [159]:
# Count total API calls
print("Total Gemini API calls:", API_CALLS)

Total Gemini API calls: 1


In [160]:
# Task 4.4 — Final recommendation and GitHub submission
# Load the required files
appointments = pd.read_csv("appointments_clean.csv")
notes = pd.read_csv("notes_analysed.csv")
print("Appointments:", appointments.shape)
print("Analysed notes:", notes.shape)

Appointments: (990, 16)
Analysed notes: (6, 8)


In [161]:
# Merge the two datasets
merged = pd.merge(
    notes,
    appointments,
    on="appointment_id",
    how="left"
)

print("Merged dataset shape:", merged.shape)

print(
    merged[
        [
            "note_id",
            "appointment_id",
            "root_cause",
            "reminder_sent",
            "status"
        ]
    ]
)

Merged dataset shape: (6, 23)
  note_id appointment_id root_cause reminder_sent   status
0    N001          A0872        NaN            No  No-Show
1    N002          A0897        NaN            No  No-Show
2    N003          A0526        NaN           Yes  No-Show
3    N004          A0926      other            No  No-Show
4    N005          A0534        NaN           Yes  No-Show
5    N006          A0507        NaN           Yes  No-Show


In [162]:
# Create a cross-tabulation
cause_reminder_table = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"]
)

print(cause_reminder_table)

reminder_sent  No
root_cause       
other           1


In [163]:
# Check the forgotten-reminder cases
forgot_cases = merged[
    merged["root_cause"] == "forgot_or_no_reminder"
]

print(forgot_cases[
    ["note_id", "appointment_id", "reminder_sent"]
])

print(
    "Forgotten/reminder cases with No reminder:",
    (forgot_cases["reminder_sent"] == "No").sum()
)

print(
    "Total forgotten/reminder cases:",
    len(forgot_cases)
)

print(
    "Total notes with No reminder:",
    (merged["reminder_sent"] == "No").sum()
)

Empty DataFrame
Columns: [note_id, appointment_id, reminder_sent]
Index: []
Forgotten/reminder cases with No reminder: 0
Total forgotten/reminder cases: 0
Total notes with No reminder: 3


In [164]:
# Save final merged dataset
merged.to_csv(
    "merged_notes_appointments.csv",
    index=False
)

print("Merged file saved!")

Merged file saved!
